# Session ID Reuse — Example Rows

## What this notebook produces

A CSV file (`session_id_sample.csv`) of real, concrete example rows
illustrating the two session-id reuse patterns identified in the companion
notebook `session_id_analysis.ipynb` in this same directory:

1. **Same-client, long-gap reuse** -- one client's `active_session_id`
   attached to demand events created many months apart (typically driven by
   a recurring autoship subscription).
2. **Cross-client collisions** -- the same `active_session_id` attached to
   demand events for genuinely different clients.

This notebook is a sampling script, not an analysis -- see
`session_id_analysis.ipynb` for the aggregate scale of each pattern and why
it matters.

In [1]:
import time

import pandas as pd
from amphibian import get_data_accessor

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

def query(sql, max_attempts=4, retry_delay_seconds=20):
    for attempt in range(1, max_attempts + 1):
        try:
            return get_data_accessor(engines=['presto']).fetch_sql(sql=sql, error_on_empty=False)
        except Exception as e:
            if attempt == max_attempts:
                raise
            print(f"query() attempt {attempt}/{max_attempts} failed ({type(e).__name__}: {e}); retrying in {retry_delay_seconds}s...")
            time.sleep(retry_delay_seconds)

START_DATE = '2025-01-01'
N_SESSION_IDS_PER_PATTERN = 5

## Pattern 1 — Same client, same session id, demand events created months apart

Picks the session ids with the largest gap between their earliest and
latest `created_ts`, restricted to a single client, then pulls every demand
row tied to each one so the full history is visible in one place.

In [2]:
pattern1_query = f"""--sql
WITH candidates AS (
    SELECT active_session_id
    FROM curated.client_reactivation_demand_events
    WHERE demand_type IN ('fix', 'direct_buy')
      AND created_ts >= DATE '{START_DATE}'
      AND active_session_id IS NOT NULL
    GROUP BY 1
    HAVING COUNT(*) >= 3
       AND COUNT(DISTINCT client_id) = 1
    ORDER BY date_diff('day', MIN(created_ts), MAX(created_ts)) DESC
    LIMIT {N_SESSION_IDS_PER_PATTERN}
)
SELECT
    de.active_session_id, de.demand_id, de.client_id, de.demand_type,
    de.fix_demand_autoship_subscription_id, de.session_datetime_in_utc, de.created_ts,
    de.client_state_detail
FROM curated.client_reactivation_demand_events de
INNER JOIN candidates c ON de.active_session_id = c.active_session_id
ORDER BY de.active_session_id, de.created_ts
"""

pattern1_df = query(pattern1_query)
pattern1_df['pattern'] = 'same_client_long_gap'
pattern1_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,active_session_id,demand_id,client_id,demand_type,fix_demand_autoship_subscription_id,session_datetime_in_utc,created_ts,client_state_detail,pattern
0,78b28115-09a2-4ad9-95d7-4af10fb571b6,609530533,4064131,fix,NaN,2025-01-07 14:35:35.032,2025-01-07 14:36:02.591,Dormant,same_client_long_gap
1,78b28115-09a2-4ad9-95d7-4af10fb571b6,609530556,4064131,fix,51220261.0,2025-01-07 14:35:35.032,2025-01-07 14:36:12.000,Dormant,same_client_long_gap
2,78b28115-09a2-4ad9-95d7-4af10fb571b6,609530555,4064131,fix,51220261.0,2025-01-07 14:35:35.032,2025-01-07 14:36:12.000,Dormant,same_client_long_gap
3,78b28115-09a2-4ad9-95d7-4af10fb571b6,609530554,4064131,fix,51220261.0,2025-01-07 14:35:35.032,2025-01-07 14:36:12.000,Dormant,same_client_long_gap
4,78b28115-09a2-4ad9-95d7-4af10fb571b6,626157619,4064131,fix,51220261.0,2025-01-07 14:35:35.032,2025-05-26 09:08:27.100,Dormant,same_client_long_gap
5,78b28115-09a2-4ad9-95d7-4af10fb571b6,637245489,4064131,fix,51220261.0,2025-01-07 14:35:35.032,2025-08-26 09:03:04.562,Dormant,same_client_long_gap
6,78b28115-09a2-4ad9-95d7-4af10fb571b6,671585066,4064131,fix,51220261.0,2025-01-07 14:35:35.032,2026-05-26 08:39:34.357,Lapsed,same_client_long_gap
7,78b28115-09a2-4ad9-95d7-4af10fb571b6,682152074,4064131,fix,51220261.0,2025-01-07 14:35:35.032,2026-08-25 09:55:21.724,Lapsed,same_client_long_gap
8,8BF1E6F8-61BA-447C-8D15-F5CED883C0EC,610301131,8443114,fix,NaN,2025-01-13 05:33:34.756,2025-01-13 05:19:05.493,Dormant,same_client_long_gap
9,8BF1E6F8-61BA-447C-8D15-F5CED883C0EC,610301174,8443114,fix,51280347.0,2025-01-13 05:33:34.756,2025-01-13 05:19:45.000,Dormant,same_client_long_gap


## Pattern 2 — Same session id, different clients

Picks session ids that appear against more than one distinct `client_id`,
then pulls every demand row tied to each one -- so it's visible whether the
different clients' rows are also close together in time (consistent with
an id-generation collision) or spread out.

In [3]:
pattern2_query = f"""--sql
WITH candidates AS (
    SELECT active_session_id
    FROM curated.client_reactivation_demand_events
    WHERE demand_type IN ('fix', 'direct_buy')
      AND created_ts >= DATE '{START_DATE}'
      AND active_session_id IS NOT NULL
    GROUP BY 1
    HAVING COUNT(DISTINCT client_id) > 1
    ORDER BY COUNT(*) DESC
    LIMIT {N_SESSION_IDS_PER_PATTERN}
)
SELECT
    de.active_session_id, de.demand_id, de.client_id, de.demand_type,
    de.fix_demand_autoship_subscription_id, de.session_datetime_in_utc, de.created_ts,
    de.client_state_detail
FROM curated.client_reactivation_demand_events de
INNER JOIN candidates c ON de.active_session_id = c.active_session_id
ORDER BY de.active_session_id, de.created_ts
"""

pattern2_df = query(pattern2_query)
pattern2_df['pattern'] = 'cross_client_collision'
pattern2_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,active_session_id,demand_id,client_id,demand_type,fix_demand_autoship_subscription_id,session_datetime_in_utc,created_ts,client_state_detail,pattern
0,4a285faf-7604-4880-89a0-735d4ca287da,631964308,9468414,fix,NaN,2025-07-13 17:13:34.159,2025-07-13 17:01:32.098,Dormant,cross_client_collision
1,4a285faf-7604-4880-89a0-735d4ca287da,631964347,9468414,fix,53048081.0,2025-07-13 17:13:34.159,2025-07-13 17:01:43.000,Dormant,cross_client_collision
2,4a285faf-7604-4880-89a0-735d4ca287da,631964346,9468414,fix,53048081.0,2025-07-13 17:13:34.159,2025-07-13 17:01:43.000,Dormant,cross_client_collision
3,4a285faf-7604-4880-89a0-735d4ca287da,631964349,9468414,fix,53048081.0,2025-07-13 17:13:34.159,2025-07-13 17:01:43.000,Dormant,cross_client_collision
4,4a285faf-7604-4880-89a0-735d4ca287da,631964345,9468414,fix,53048081.0,2025-07-13 17:13:34.159,2025-07-13 17:01:43.000,Dormant,cross_client_collision
...,...,...,...,...,...,...,...,...,...
312,ec72de6c-3cea-4662-858b-3ec68a8f8f65,658047647,46812529,fix,55241694.0,2026-02-11 10:09:08.826,2026-02-11 10:04:30.000,Dormant,cross_client_collision
313,ec72de6c-3cea-4662-858b-3ec68a8f8f65,658047645,46812529,fix,55241694.0,2026-02-11 10:09:08.826,2026-02-11 10:04:30.000,Dormant,cross_client_collision
314,ec72de6c-3cea-4662-858b-3ec68a8f8f65,658047655,46812529,fix,55241694.0,2026-02-11 10:09:08.826,2026-02-11 10:04:30.000,Dormant,cross_client_collision
315,ec72de6c-3cea-4662-858b-3ec68a8f8f65,658047652,46812529,fix,55241694.0,2026-02-11 10:09:08.826,2026-02-11 10:04:30.000,Dormant,cross_client_collision


## Write the combined sample CSV

In [4]:
sample_df = pd.concat([pattern1_df, pattern2_df], ignore_index=True)
sample_df.to_csv("session_id_sample.csv", index=False)
print(f"Wrote {len(sample_df)} rows to session_id_sample.csv")
sample_df

Wrote 362 rows to session_id_sample.csv


,active_session_id,demand_id,client_id,demand_type,fix_demand_autoship_subscription_id,session_datetime_in_utc,created_ts,client_state_detail,pattern
0,78b28115-09a2-4ad9-95d7-4af10fb571b6,609530533,4064131,fix,NaN,2025-01-07 14:35:35.032,2025-01-07 14:36:02.591,Dormant,same_client_long_gap
1,78b28115-09a2-4ad9-95d7-4af10fb571b6,609530556,4064131,fix,51220261.0,2025-01-07 14:35:35.032,2025-01-07 14:36:12.000,Dormant,same_client_long_gap
2,78b28115-09a2-4ad9-95d7-4af10fb571b6,609530555,4064131,fix,51220261.0,2025-01-07 14:35:35.032,2025-01-07 14:36:12.000,Dormant,same_client_long_gap
3,78b28115-09a2-4ad9-95d7-4af10fb571b6,609530554,4064131,fix,51220261.0,2025-01-07 14:35:35.032,2025-01-07 14:36:12.000,Dormant,same_client_long_gap
4,78b28115-09a2-4ad9-95d7-4af10fb571b6,626157619,4064131,fix,51220261.0,2025-01-07 14:35:35.032,2025-05-26 09:08:27.100,Dormant,same_client_long_gap
...,...,...,...,...,...,...,...,...,...
357,ec72de6c-3cea-4662-858b-3ec68a8f8f65,658047647,46812529,fix,55241694.0,2026-02-11 10:09:08.826,2026-02-11 10:04:30.000,Dormant,cross_client_collision
358,ec72de6c-3cea-4662-858b-3ec68a8f8f65,658047645,46812529,fix,55241694.0,2026-02-11 10:09:08.826,2026-02-11 10:04:30.000,Dormant,cross_client_collision
359,ec72de6c-3cea-4662-858b-3ec68a8f8f65,658047655,46812529,fix,55241694.0,2026-02-11 10:09:08.826,2026-02-11 10:04:30.000,Dormant,cross_client_collision
360,ec72de6c-3cea-4662-858b-3ec68a8f8f65,658047652,46812529,fix,55241694.0,2026-02-11 10:09:08.826,2026-02-11 10:04:30.000,Dormant,cross_client_collision
